# Maia Lite 0.3.0 — Exportação final SFT Full-Linear → Hugging Face GPT-2 → GGUF

**Somente exportação, sem treinamento.** Base **step 300000**, LoRA **update 5744**, arquitetura GPT-2-like (36 camadas, 1024 dimensões, 16 cabeças, contexto 2048, vocabulário 50257). Executar no **Google Colab** com Google Drive.

Pipeline: verificar origem → merge LoRA em CPU FP32 → mapear pesos para `GPT2LMHeadModel` → salvar `safetensors` e tokenizer → validar equivalência numérica → conversão com `llama.cpp` → smoke test GGUF → ZIP para servidor.

**Atenção:** compatibilidade de GPT-2 no conversor e no Ollama depende das versões instaladas. O notebook verifica os comandos e interrompe se não houver suporte; não afirma que um GGUF gerado necessariamente será aceito pelo Ollama. Execute apenas com checkpoints próprios e confiáveis (`torch.load` usa pickle).

## 1 — Preparação e caminhos

In [ ]:
import transformers
import huggingface_hub

print("Transformers:", transformers.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)

Transformers: 4.57.6
Hugging Face Hub: 0.36.2


In [ ]:
from transformers import GPT2Config, GPT2LMHeadModel
from huggingface_hub import hf_hub_download
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("Transformers e Hugging Face Hub: OK")

PyTorch: 2.14.1+cu130
CUDA: True
Transformers e Hugging Face Hub: OK


In [ ]:
#%pip -q install 'transformers>=4.44,<5' 'safetensors>=0.4' 'accelerate>=0.33' 'sentencepiece' 'protobuf'

In [6]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import torch, json, hashlib, gc, shutil, subprocess, sys, os, re, time
from safetensors.torch import save_file, load_file
from transformers import AutoTokenizer, GPT2Config, GPT2LMHeadModel
ROOT=Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat')
BASE_CKPT=ROOT/'training_runs/maia_lite_0_3_0_a100_v3_experimental/latest.pt'
ADAPTER_CKPT=ROOT/'training_runs/maia_lite_0_3_0_sft_full_linear_v2/adapter_latest.pt'
TOKENIZER_DIR=Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3/tokenizer')
OUT=ROOT/'exports/maia_lite_0_3_0_full_linear_final'
HF_DIR=OUT/'hf_fp16'; GGUF=OUT/'maia-lite-0.3.0-sft-f16.gguf'
EXPECTED_BASE_STEP=300000; EXPECTED_UPDATE=5744
for label,p in [('base',BASE_CKPT),('adapter',ADAPTER_CKPT)]:
    assert p.is_file(),f'{label} não encontrado: {p}'
assert TOKENIZER_DIR.is_dir(),TOKENIZER_DIR
OUT.mkdir(parents=True,exist_ok=True)
print('Base:',BASE_CKPT,'\nAdapter:',ADAPTER_CKPT,'\nSaída:',OUT)


Mounted at /content/drive
Base: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/training_runs/maia_lite_0_3_0_a100_v3_experimental/latest.pt 
Adapter: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/training_runs/maia_lite_0_3_0_sft_full_linear_v2/adapter_latest.pt 
Saída: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/exports/maia_lite_0_3_0_full_linear_final


## 2 — Auditoria de checkpoints e metadados

In [ ]:
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(8*1024*1024),b''):h.update(block)
    return h.hexdigest()
base=torch.load(BASE_CKPT,map_location='cpu',weights_only=False)
adapt=torch.load(ADAPTER_CKPT,map_location='cpu',weights_only=False)
assert isinstance(base,dict) and isinstance(base.get('model'),dict)
assert int(base.get('step',-1))==EXPECTED_BASE_STEP,base.get('step')
assert isinstance(adapt.get('adapter'),dict) and isinstance(adapt.get('config'),dict)
assert int(adapt.get('update',-1))==EXPECTED_UPDATE,adapt.get('update')
ac=adapt['config']
assert int(ac['rank'])==16 and int(ac['alpha'])==32
assert ac['targets']==['qkv','proj','fc','out']
assert int(ac.get('base_step',-1))==EXPECTED_BASE_STEP
print('Base e adapter confirmados. Hashes:')
print('base:',sha256(BASE_CKPT));print('adapter:',sha256(ADAPTER_CKPT))
print('Pesos base:',len(base['model']),'LoRA:',len(adapt['adapter']))


Base e adapter confirmados. Hashes:
base: 332be4064daa108588488bb54cefb95ec69233d92e027f0a81e2e3500b380d1d
adapter: c3c6c18fb8b8608a28ec59be9dfc7a131c91cffec71b4f5065ba7054497a3638
Pesos base: 437 LoRA: 288


## 3 — Merge dos adaptadores LoRA (CPU, FP32)

A transformação usada no treino é `W_merged = W_base + (alpha/rank) * (B @ A)`. A inferência LoRA do treino calcula o incremento em FP32 e converte para BF16; por isso, diferenças numéricas pequenas são esperadas. Não há dropout na inferência.

In [ ]:
base_sd=base['model']; lora=adapt['adapter']; merged={}
expected=set()
for layer in range(36):
    for target in ('qkv','proj','fc','out'):
        prefix=f'blocks.{layer}.{target}'
        expected.update((prefix+'.A',prefix+'.B'))
assert set(lora)==expected,('Adaptador incompleto ou extra',len(expected-set(lora)),len(set(lora)-expected))
for key,w in base_sd.items():
    if key.endswith('.weight'):
        prefix=key[:-7]
        if prefix+'.A' in lora:
            A=lora[prefix+'.A'].float(); B=lora[prefix+'.B'].float()
            assert w.ndim==2 and B.shape[0]==w.shape[0] and A.shape[1]==w.shape[1]
            merged[key]=w.float() + (32.0/16.0)*(B@A)
            continue
    merged[key]=w.float() if w.is_floating_point() else w.clone()
assert len(merged)==len(base_sd)
print('Merge realizado:',len(merged),'tensores; parâmetros ajustados:',len(expected)//2)
del base,adapt,base_sd,lora;gc.collect()


Merge realizado: 437 tensores; parâmetros ajustados: 144


100

## 4 — Mapear a arquitetura Maia para GPT-2 Hugging Face

`nn.Linear` usa matriz `[out,in]`, enquanto `transformers.GPT2` usa `Conv1D` com `[in,out]`. As projeções de atenção e MLP são transpostas. Embeddings e LayerNorm permanecem no formato original.

In [ ]:
cfg=GPT2Config(vocab_size=50257,n_positions=2048,n_ctx=2048,n_embd=1024,n_layer=36,n_head=16,
               resid_pdrop=0.0,embd_pdrop=0.0,attn_pdrop=0.0,activation_function='gelu_new',
               layer_norm_epsilon=1e-5,bos_token_id=50256,eos_token_id=50256,tie_word_embeddings=True)
converted={
 'transformer.wte.weight':merged['tok.weight'],
 'transformer.wpe.weight':merged['pos.weight'],
 'transformer.ln_f.weight':merged['norm.weight'],
 'transformer.ln_f.bias':merged['norm.bias'],
}
for i in range(36):
    p=f'blocks.{i}.'; q=f'transformer.h.{i}.'
    for old,new in [('ln1','ln_1'),('ln2','ln_2')]:
        for v in ('weight','bias'):converted[q+new+'.'+v]=merged[p+old+'.'+v]
    for old,new in [('qkv','attn.c_attn'),('proj','attn.c_proj'),('fc','mlp.c_fc'),('out','mlp.c_proj')]:
        for v in ('weight','bias'):
            w=merged[p+old+'.'+v]
            converted[q+new+'.'+v]=w.T.contiguous() if v=='weight' else w
# A cabeça está ligada ao embedding, não deve ser salva como tensor duplicado no safetensors.
assert torch.equal(merged['tok.weight'],merged['head.weight']), 'Embedding/head não estão tied: revisar arquitetura'
print('Tensores HF:',len(converted))


Tensores HF: 436


## 5 — Validação numérica da conversão antes de salvar

Carrega o modelo Hugging Face em CPU para verificar nomes, shapes e logits em sequências curtas. Usa FP32 nesta validação para isolar erros de mapeamento. A referência Maia é calculada diretamente com os tensores mesclados.

In [ ]:
import torch.nn.functional as F
with torch.device('meta'):
    hf=GPT2LMHeadModel(cfg)
missing,unexpected=hf.load_state_dict(converted,strict=False,assign=True)
assert not unexpected,unexpected
assert set(missing).issubset({'lm_head.weight'}),missing
hf.tie_weights();hf.eval()
@torch.inference_mode()
def maia_logits(ids):
    x=F.embedding(ids,merged['tok.weight'])+F.embedding(torch.arange(ids.size(1)),merged['pos.weight'])
    for i in range(36):
        p=f'blocks.{i}.'
        def ln(z,name):return F.layer_norm(z,(1024,),merged[p+name+'.weight'],merged[p+name+'.bias'],1e-5)
        def lin(z,name):return F.linear(z,merged[p+name+'.weight'],merged[p+name+'.bias'])
        q,k,v=lin(ln(x,'ln1'),'qkv').chunk(3,-1)
        def split(t):return t.view(t.size(0),t.size(1),16,64).transpose(1,2)
        a=F.scaled_dot_product_attention(split(q),split(k),split(v),is_causal=True,dropout_p=0.0)
        x=x+lin(a.transpose(1,2).contiguous().view(x.size(0),x.size(1),1024),'proj')
        x=x+lin(F.gelu(lin(ln(x,'ln2'),'fc'),approximate='tanh'),'out')
    x=F.layer_norm(x,(1024,),merged['norm.weight'],merged['norm.bias'],1e-5)
    return F.linear(x,merged['tok.weight'])
ids=torch.tensor([[100,200,300,400]],dtype=torch.long)
with torch.inference_mode():
    ref=maia_logits(ids)
    got=hf(ids).logits
maxdiff=(ref-got).abs().max().item()
print('Diferença máxima logits FP32:',maxdiff)
assert torch.allclose(ref,got,rtol=2e-3,atol=2e-2),f'Mapeamento divergente: {maxdiff}'
print('PASS: equivalência numérica GPT-2')


Diferença máxima logits FP32: 3.814697265625e-05
PASS: equivalência numérica GPT-2


## 6 — Salvar modelo Hugging Face e tokenizer

Salva pesos FP16 em `safetensors` e `config.json`. O checkpoint original e o adaptador não são alterados.

In [ ]:
HF_DIR.mkdir(parents=True,exist_ok=True)
tokenizer=AutoTokenizer.from_pretrained(str(TOKENIZER_DIR),local_files_only=True,use_fast=True)
assert len(tokenizer)==50257 and tokenizer.eos_token_id is not None
cfg.bos_token_id=tokenizer.bos_token_id if tokenizer.bos_token_id is not None else tokenizer.eos_token_id
cfg.eos_token_id=tokenizer.eos_token_id
hf.config=cfg
hf.half()
hf.save_pretrained(HF_DIR,safe_serialization=True,max_shard_size='2GB')
tokenizer.save_pretrained(HF_DIR)
print('Salvo:',HF_DIR)
# Verificação de reabertura
from transformers import GPT2LMHeadModel as GPT2
check=GPT2.from_pretrained(HF_DIR,local_files_only=True,torch_dtype=torch.float16).eval()
with torch.inference_mode():check_logits=check(ids).logits.float()
print('Diferença FP16 vs FP32:',(ref-check_logits).abs().max().item())
del check,hf,merged,converted,ref,got,check_logits;gc.collect()


`torch_dtype` is deprecated! Use `dtype` instead!


Salvo: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/exports/maia_lite_0_3_0_full_linear_final/hf_fp16
Diferença FP16 vs FP32: 1.1296491622924805


12

## 7 — Preparar llama.cpp e converter para GGUF F16

A compatibilidade de GPT-2 deve ser verificada no `convert_hf_to_gguf.py` da versão clonada. Se o conversor não reconhecer `GPT2LMHeadModel`, **pare** e ajuste o suporte de arquitetura; não renomeie a arquitetura como Llama.

In [ ]:
LLAMA=Path('/content/llama.cpp')
if not LLAMA.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/ggml-org/llama.cpp',str(LLAMA)],check=True)
print('Commit:',subprocess.check_output(['git','-C',str(LLAMA),'rev-parse','HEAD'],text=True).strip())
script=LLAMA/'convert_hf_to_gguf.py'
assert script.exists(),script
helptext=subprocess.run([sys.executable,str(script),'--help'],capture_output=True,text=True)
assert helptext.returncode==0,helptext.stderr[-3000:]
print(helptext.stdout[-1800:])


Commit: e6d624b9a4b95c6b52f5ca384d70b75581f9821a
4 or Mistral Large 4 DSpark
                        draft tensors as a separate GGUF.
  --mistral-format      Whether the model is stored following the Mistral
                        format.
  --disable-mistral-community-chat-template
                        Whether to disable usage of Mistral community chat
                        templates. If set, use the Mistral official `mistral-
                        common` library for tokenization and detokenization of
                        Mistral models. Using `mistral-common` ensure
                        correctness and zero-day support of tokenization for
                        models converted from the Mistral format but requires
                        to manually setup the tokenization server.
  --sentence-transformers-dense-modules
                        Whether to include sentence-transformers dense
                        modules. It can be used for sentence-transformers
      

In [ ]:
from pathlib import Path

script = Path("/content/llama.cpp/conversion/base.py")
source = script.read_text()

target = (
    'raise NotImplementedError('
    '"BPE pre-tokenizer was not recognized - update get_vocab_base_pre()")'
)

patch = '''
if chkhsh == "a727209002a4bef5c366d1ebedf90abc1d4c85d919dfa18a2364d27747dcb599":
            return "gpt-2"
        '''

# Verificar a implementação antes de alterar
start = source.find("def get_vocab_base_pre(")
end = source.find("\n    def ", start + 5)

assert start >= 0, "Função não encontrada"
function = source[start:end if end > start else None]

print(function[-3500:])

afc8b9c40eab6ea89ece4ec02fbc57":
            # ref: https://huggingface.co/sarvamai/sarvam-30b
            res = "sarvam-moe"
        if chkhsh == "f728162c1315c26e40249849799b4ba3fe584c32084b4795b03eb295e63cb5af":
            # ref: https://huggingface.co/lewtun/talkie-1930-13b-it-hf
            res = "talkie"
        if chkhsh == "36f3066e97b7f3994b379aaacde306c1444c6ae84e81a5ae3cd2b7ed3b8c42d4":
            # ref: https://huggingface.co/openbmb/MiniCPM5-1B
            res = "minicpm5"
        if chkhsh == "f241072145675bf8322086f115aebad05e9f869557a238bf2150a2a417d1bf60":
            # ref: https://huggingface.co/ibm-granite/granite-embedding-97m-multilingual-r2
            res = "granite-embed-multi-97m"
        if chkhsh == "789696f5946cc0fc59371f39f6097cafed196b3acded6140432f26bbb1ae1669":
            # ref: https://huggingface.co/ibm-granite/granite-embedding-311m-multilingual-r2
            res = "granite-embed-multi-311m"
        if chkhsh == "9dcf830ee9990cdbf78cc523a5f7bd9ad

In [ ]:
from pathlib import Path

path = Path("/content/llama.cpp/conversion/base.py")
source = path.read_text()

hash_maia = "a727209002a4bef5c366d1ebedf90abc1d4c85d919dfa18a2364d27747dcb599"

marker = "        if res is None:\n"

patch = f'''        # Maia Lite 0.3.0 - GPT-2 ByteLevel BPE
        if chkhsh == "{hash_maia}":
            res = "gpt-2"

'''

assert marker in source, "Ponto de inserção não encontrado"

if f'if chkhsh == "{hash_maia}"' not in source:
    source = source.replace(marker, patch + marker, 1)
    path.write_text(source)
    print("Patch Maia Lite 0.3.0 aplicado.")
else:
    print("Patch já presente.")

import py_compile
py_compile.compile(str(path), doraise=True)

print("Sintaxe Python: OK")

Patch Maia Lite 0.3.0 aplicado.
Sintaxe Python: OK


In [ ]:
cmd=[sys.executable,str(script),str(HF_DIR),'--outfile',str(GGUF),'--outtype','f16']
print('Executando:', ' '.join(cmd))
run=subprocess.run(cmd,cwd=LLAMA,text=True,capture_output=True)
print(run.stdout[-5000:]);print(run.stderr[-5000:])
if run.returncode!=0:
    raise RuntimeError('Conversão falhou. Verifique suporte GPT-2 na versão llama.cpp e os logs acima. Não há GGUF validado.')
assert GGUF.is_file() and GGUF.stat().st_size>100_000_000
print('GGUF:',GGUF,'bytes:',GGUF.stat().st_size,'sha256:',sha256(GGUF))


Executando: /usr/bin/python3 /content/llama.cpp/conversion/base.py /content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/exports/maia_lite_0_3_0_full_linear_final/hf_fp16 --outfile /content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/exports/maia_lite_0_3_0_full_linear_final/maia-lite-0.3.0-sft-f16.gguf --outtype f16




AssertionError: 

In [ ]:
from pathlib import Path
import json

HF = Path(
    "/content/drive/MyDrive/Colab_LLMs/"
    "Maia_Lite_0_3_0_Chat/exports/"
    "maia_lite_0_3_0_full_linear_final/hf_fp16"
)

for name in [
    "tokenizer_config.json",
    "tokenizer.json",
    "config.json"
]:
    path = HF / name
    print(f"\n=== {name} ===")

    if not path.exists():
        print("ARQUIVO AUSENTE")
        continue

    data = json.loads(path.read_text())

    if name == "tokenizer.json":
        print("Pre-tokenizer:")
        print(json.dumps(
            data.get("pre_tokenizer"),
            indent=2,
            ensure_ascii=False
        ))
        print("Modelo tokenizer:", data.get("model", {}).get("type"))
        print("Vocabulário:", len(data.get("model", {}).get("vocab", {})))
    else:
        print(json.dumps(data, indent=2, ensure_ascii=False)[:3000])


=== tokenizer_config.json ===
{
  "add_prefix_space": false,
  "added_tokens_decoder": {
    "0": {
      "content": "<s>",
      "lstrip": false,
      "normalized": false,
      "rstrip": false,
      "single_word": false,
      "special": true
    },
    "1": {
      "content": "<pad>",
      "lstrip": false,
      "normalized": false,
      "rstrip": false,
      "single_word": false,
      "special": true
    },
    "2": {
      "content": "</s>",
      "lstrip": false,
      "normalized": false,
      "rstrip": false,
      "single_word": false,
      "special": true
    },
    "3": {
      "content": "<unk>",
      "lstrip": false,
      "normalized": false,
      "rstrip": false,
      "single_word": false,
      "special": true
    },
    "4": {
      "content": "<mask>",
      "lstrip": false,
      "normalized": false,
      "rstrip": false,
      "single_word": false,
      "special": true
    }
  },
  "backend": "tokenizers",
  "bos_token": "<s>",
  "clean_up_tokenization

In [ ]:
import subprocess
import sys
from pathlib import Path

HF = Path(
    "/content/drive/MyDrive/Colab_LLMs/"
    "Maia_Lite_0_3_0_Chat/exports/"
    "maia_lite_0_3_0_full_linear_final/hf_fp16"
)

GGUF = HF.parent / "maia-lite-0.3.0-sft-f16.gguf"

CONVERTER = Path("/content/llama.cpp/convert_hf_to_gguf.py")

assert CONVERTER.is_file(), "Conversor não encontrado"
assert HF.is_dir(), "Modelo Hugging Face não encontrado"

cmd = [
    sys.executable,
    str(CONVERTER),
    str(HF),
    "--outfile", str(GGUF),
    "--outtype", "f16",
]

print("Executando:", " ".join(cmd), flush=True)

run = subprocess.run(
    cmd,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print(run.stdout[-12000:])

if run.returncode != 0:
    raise RuntimeError(
        f"Conversão falhou (código {run.returncode})."
    )

assert GGUF.is_file(), "GGUF não foi gerado"
assert GGUF.stat().st_size > 100_000_000, "GGUF incompleto"

print("\nConversão concluída!")
print("Arquivo:", GGUF)
print("Tamanho:", round(GGUF.stat().st_size / 1024**3, 3), "GiB")

Executando: /usr/bin/python3 /content/llama.cpp/convert_hf_to_gguf.py /content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/exports/maia_lite_0_3_0_full_linear_final/hf_fp16 --outfile /content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/exports/maia_lite_0_3_0_full_linear_final/maia-lite-0.3.0-sft-f16.gguf --outtype f16
shape = {1024}
INFO:hf-to-gguf:blk.6.ffn_norm.bias,       torch.float16 --> F32, shape = {1024}
INFO:hf-to-gguf:blk.6.ffn_norm.weight,     torch.float16 --> F32, shape = {1024}
INFO:hf-to-gguf:blk.6.ffn_up.bias,         torch.float16 --> F32, shape = {4096}
INFO:hf-to-gguf:blk.6.ffn_up.weight,       torch.float16 --> F16, shape = {1024, 4096}
INFO:hf-to-gguf:blk.6.ffn_down.bias,       torch.float16 --> F32, shape = {1024}
INFO:hf-to-gguf:blk.6.ffn_down.weight,     torch.float16 --> F16, shape = {4096, 1024}
INFO:hf-to-gguf:blk.7.attn_qkv.bias,       torch.float16 --> F32, shape = {3072}
INFO:hf-to-gguf:blk.7.attn_qkv.weight,     torch.float16 --> F16, shape = {1024, 

## 8 — Smoke test no llama.cpp (obrigatório)

O teste usa CPU e uma única pergunta. Se a compilação ou execução falhar, **não publique o GGUF**. Algumas versões de `llama.cpp` podem não oferecer inferência GPT-2, mesmo quando conseguem converter os pesos.

In [ ]:
build=LLAMA/'build'
subprocess.run(['cmake','-S',str(LLAMA),'-B',str(build),'-DCMAKE_BUILD_TYPE=Release','-DGGML_CUDA=OFF'],check=True)
subprocess.run(['cmake','--build',str(build),'--config','Release','-j','2','--target','llama-cli'],check=True)
cli=build/'bin/llama-cli';assert cli.is_file(),cli
prompt='Usuário: Qual é a capital do Brasil?\nAssistente:\n'
run=subprocess.run([str(cli),'-m',str(GGUF),'-p',prompt,'-n','48','--temp','0','-c','2048'],capture_output=True,text=True,timeout=300)
print(run.stdout[-5000:]);print(run.stderr[-3000:])
assert run.returncode==0,'Falha na inferência GGUF. Não publicar até investigar.'
print('PASS: llama.cpp carregou e executou o GGUF (a qualidade da resposta exige revisão humana).')




Loading model... |-\|/-\|/-\|/ 

▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b1-e6d624b
model      : /content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/exports/maia_lite_0_3_0_full_linear_final/maia-lite-0.3.0-sft-f16.gguf
ftype      : F16
modalities : text

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
  /read <file>        add a text file
  /glob <pattern>     add text files using globbing pattern



> Usuário: Qual é a capital do Brasil?
Assistente:

<|im_end|>user
<|im_start|>user
<|im_end|>user
<|im_end|>user
<|im_end|>user
<|im

[ Prompt: 280.5 t/s | Generation: 45.1 t/s ]

> 

Exiting...


PASS: llama.cpp carregou e executou o GGUF (a qualidade

## 9 — Empacotar para o servidor e registrar proveniência

ZIP inclui GGUF, Modelfile sugerido, README e manifest. O modelo Ollama só deve ser criado após testar se a versão instalada suporta a arquitetura GPT-2 GGUF. **Não assume suporte automático**.

In [7]:
required = [
    "EXPECTED_BASE_STEP",
    "EXPECTED_UPDATE",
    "BASE_CKPT",
    "ADAPTER_CKPT",
    "GGUF",
    "HF_DIR",
    "LLAMA",
    "sha256",
]

for name in required:
    print(
        f"{name:22}",
        "OK" if name in globals() else "NÃO DEFINIDA"
    )

EXPECTED_BASE_STEP     OK
EXPECTED_UPDATE        OK
BASE_CKPT              OK
ADAPTER_CKPT           OK
GGUF                   OK
HF_DIR                 OK
LLAMA                  NÃO DEFINIDA
sha256                 NÃO DEFINIDA


In [9]:
from pathlib import Path
import hashlib
import subprocess
import json

def sha256(path, chunk_size=8 * 1024 * 1024):
    """Calcula SHA-256 de arquivos grandes sem carregá-los na RAM."""
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)

    return h.hexdigest()

# Diretórios
LLAMA = Path("/content/llama.cpp")

HF_DIR = Path(
    "/content/drive/MyDrive/Colab_LLMs/"
    "Maia_Lite_0_3_0_Chat/exports/"
    "maia_lite_0_3_0_full_linear_final/hf_fp16"
)

GGUF = HF_DIR.parent / "maia-lite-0.3.0-sft-f16.gguf"

# Metadados
EXPECTED_BASE_STEP = 300000
EXPECTED_UPDATE = 5744

assert GGUF.is_file(), f"GGUF não encontrado: {GGUF}"

print("GGUF encontrado:", GGUF)
print("Tamanho:", round(GGUF.stat().st_size / 1024**3, 3), "GiB")
print("Função SHA-256 disponível.")

GGUF encontrado: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/exports/maia_lite_0_3_0_full_linear_final/maia-lite-0.3.0-sft-f16.gguf
Tamanho: 0.951 GiB
Função SHA-256 disponível.


In [11]:
import json
import subprocess
from datetime import datetime, timezone

def get_git_commit(repo):
    result = subprocess.run(
        ["git", "-C", str(repo), "rev-parse", "HEAD"],
        capture_output=True,
        text=True
    )
    return result.stdout.strip() if result.returncode == 0 else None

assert GGUF.is_file(), "GGUF não encontrado"

manifest = {
    "model": "Maia Lite 0.3.0 SFT Full-Linear",
    "architecture": "GPT-2-like",
    "base_step": EXPECTED_BASE_STEP,
    "adapter_update": EXPECTED_UPDATE,
    "base_sha256": sha256(BASE_CKPT),
    "adapter_sha256": sha256(ADAPTER_CKPT),
    "gguf_sha256": sha256(GGUF),
    "gguf_bytes": GGUF.stat().st_size,
    "hf_dir": str(HF_DIR),
    "llama_cpp_commit": get_git_commit(LLAMA),
    "prompt_template": "Usuário: {question}\nAssistente:\n",
    "conversion": "merged LoRA rank16 alpha32 + GPT2 HF + llama.cpp f16",
    "created_utc": datetime.now(timezone.utc).isoformat()
}

manifest_path = GGUF.parent / "manifest.json"

manifest_path.write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("Manifesto salvo:", manifest_path)
print("GGUF:", GGUF.name)
print("Tamanho:", round(GGUF.stat().st_size / 1024**3, 3), "GiB")
print("SHA-256:", manifest["gguf_sha256"])
print("Commit llama.cpp:", manifest["llama_cpp_commit"] or "não disponível")

Manifesto salvo: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/exports/maia_lite_0_3_0_full_linear_final/manifest.json
GGUF: maia-lite-0.3.0-sft-f16.gguf
Tamanho: 0.951 GiB
SHA-256: 340c7cde71dfc69f2f2a339b0d85db9a7c078447e85d4ae50ab679c4539042a5
Commit llama.cpp: não disponível


## 10 — Servidor Maia Chat

Copiar o ZIP do Drive para o servidor e descompactar. Verificar hash do GGUF. Testar `llama-cli` no servidor, e somente depois `ollama create maia-lite:0.3.0 -f Modelfile` e `ollama run maia-lite:0.3.0` **se o Ollama instalado suportar GPT-2 GGUF**. Se não suportar, registrar a incompatibilidade e usar um backend `llama.cpp` que suporte a arquitetura ou desenvolver suporte; não forçar uma conversão para Llama.

A quantização (`Q8_0`/`Q4_K_M`) é **opcional** e deve ser feita somente após o F16 passar nos testes de inferência e equivalência.

**Critério de fechamento:** hashes + validação dos logits HF + GGUF inferindo + teste no servidor. Depois iniciamos o Maia Lite 0.4.0 ~355M.